# Wrangle Meteo Data

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
cities_df = pd.read_csv('meteo_colombia/cities_geolocated.csv')
cities_df.head()

,city,state,country,Latitude,Longitude,idx_city
0,exterior_aruba,exterior,aruba,12.501363,-69.961848,ARUBA_EXTERIOR_EXTERIOR_ARUBA
1,exterior_brasil,exterior,brasil,-10.333333,-53.200000,BRASIL_EXTERIOR_EXTERIOR_BRASIL
2,leticia,amazonas,colombia,-4.212921,-69.942596,COLOMBIA_AMAZONAS_LETICIA
3,apartado,antioquia,colombia,7.884901,-76.622746,COLOMBIA_ANTIOQUIA_APARTADO
4,caracoli,antioquia,colombia,6.409276,-74.756698,COLOMBIA_ANTIOQUIA_CARACOLI


In [3]:
import requests

import math
import time   # only needed if you want to be really gentle to the free service

# --- helper ---------------------------------------------------------------
def fetch_elevation_opentopo(latitudes, longitudes, dem="srtm90m"):
    """
    Query OpenTopoData for a list/array of latitudes and longitudes.
    Returns a list of elevations (metres).  Order is preserved.
    """
    elevations = []
    BATCH = 100                         # OpenTopoData allows ~100–200 per call comfortably
    for i in range(0, len(latitudes), BATCH):
        locs = "|".join(
            f"{lat},{lon}"
            for lat, lon in zip(latitudes[i:i+BATCH], longitudes[i:i+BATCH])
        )
        url = f"https://api.opentopodata.org/v1/{dem}?locations={locs}"
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        elevations.extend([item["elevation"] for item in data["results"]])
        # optional politeness pause to avoid hammering the public server
        time.sleep(0.2)
    return elevations
# --------------------------------------------------------------------------

# existing df with 'latitude' and 'longitude'
cities_df["elevation_m"] = fetch_elevation_opentopo(cities_df["Latitude"].values,
                                             cities_df["Longitude"].values)



HTTPError: 429 Client Error: Too Many Requests for url: https://api.opentopodata.org/v1/srtm90m?locations=8.8746769,-75.6210036%7C5.3461338,-74.4916464%7C10.6284107,-72.87309324366868%7C1.8070452,-75.888501%7C4.5664176,-75.7507081%7C3.96594345,-74.72676148715544%7C5.5511075,-69.61626306881838%7C3.1323331,-76.3916018%7C2.7705138,-77.6643122%7C8.3912927,-73.3815389%7C8.0019296,-73.5121466%7C4.5181995,-74.7899996%7C4.4388882,-74.5233576%7C10.0269357,-74.6208028%7C3.7393505,-73.8350537%7C2.9931252,-76.4498155%7C8.3546568,-72.4145867%7C4.5305516,-75.6406048%7C8.851812,-75.2804647%7C3.971341,-75.51451613072902%7C-1.3183166,-69.5863801%7C6.3265312,-75.7685045%7C5.98430825,-75.44752287622777%7C6.4427228,-75.727937%7C8.9566235,-74.0770262%7C4.998398699999999,-73.12033660522744%7C5.0743694,-75.50811667440546%7C2.4422295,-76.6072368%7C5.54747405,-76.64208722719327%7C5.00335,-76.0031995%7C6.2881742,-73.148289%7C6.4370185,-75.3313779%7C6.2954934,-75.0280123%7C10.41639435,-74.85603351926324%7C5.1708

In [4]:
cities_df

,city,state,country,Latitude,Longitude,idx_city
0,exterior_aruba,exterior,aruba,12.501363,-69.961848,ARUBA_EXTERIOR_EXTERIOR_ARUBA
1,exterior_brasil,exterior,brasil,-10.333333,-53.200000,BRASIL_EXTERIOR_EXTERIOR_BRASIL
2,leticia,amazonas,colombia,-4.212921,-69.942596,COLOMBIA_AMAZONAS_LETICIA
3,apartado,antioquia,colombia,7.884901,-76.622746,COLOMBIA_ANTIOQUIA_APARTADO
4,caracoli,antioquia,colombia,6.409276,-74.756698,COLOMBIA_ANTIOQUIA_CARACOLI
...,...,...,...,...,...,...
797,teruel,huila,colombia,2.741633,-75.568345,COLOMBIA_HUILA_TERUEL
798,buenavista,quindio,colombia,4.359948,-75.738723,COLOMBIA_QUINDIO_BUENAVISTA
799,belen de umbria,risaralda,colombia,5.200909,-75.868993,COLOMBIA_RISARALDA_BELEN DE UMBRIA
800,exterior_comoras,exterior,comoras,-12.204518,44.283296,COMORAS_EXTERIOR_EXTERIOR_COMORAS


In [5]:
# get all csv files in meteo_colombia/cities path
path = 'meteo_colombia/cities/'
files = os.listdir(path)


In [6]:
df_t = pd.read_csv(path + files[0])
len(df_t)

0

In [7]:
dict_empty_files = {}
for file in files:
    df_file = pd.read_csv(f'meteo_colombia/cities/{file}')
    # check if the file is empty
    if len(df_file) == 0:
        dict_empty_files[file] = 'empty'
    else:
        dict_empty_files[file] = 'not_empty'


In [8]:
# dict_empty_files to dataframe
df_empty_files = pd.DataFrame.from_dict(dict_empty_files, orient='index', columns=['status']).reset_index()
df_empty_files.rename(columns={'index': 'idx_city'}, inplace=True)
df_empty_files['idx_city'] = df_empty_files['idx_city'].str.replace('.csv', '')
df_empty_files

,idx_city,status
0,ARGENTINA_EXTERIOR_EXTERIOR_ARGENTINA,empty
1,ARUBA_EXTERIOR_EXTERIOR_ARUBA,not_empty
2,BAHAMAS_EXTERIOR_EXTERIOR_BAHAMAS,empty
3,BOLIVIA_EXTERIOR_EXTERIOR_BOLIVIA,empty
4,BRASIL_EXTERIOR_EXTERIOR_BRASIL,empty
...,...,...
797,MÉXICO_EXTERIOR_EXTERIOR_MÉXICO,empty
798,PANAMÁ_EXTERIOR_EXTERIOR_PANAMA,empty
799,PERÚ_EXTERIOR_EXTERIOR_PERU,empty
800,REPÚBLICA DOMINICANA_EXTERIOR_EXTERIOR_REPÚBLI...,empty


In [9]:
# merge df_empty_files with cities_df
df_merge = pd.merge(cities_df, df_empty_files, on='idx_city', how='left')

In [10]:
df_empty = df_merge[df_merge['status'] == 'empty'].copy()
df_not_empty = df_merge[df_merge['status'] == 'not_empty'].copy()

In [11]:
print(len(df_empty), 'Empty files')
print(len(df_not_empty), 'Not empty files')

532 Empty files
270 Not empty files


In [12]:
df_empty_col = df_empty[df_empty['country'] == 'colombia'].copy().reset_index(drop=True)
df_empty_col

,city,state,country,Latitude,Longitude,idx_city,status
0,caracoli,antioquia,colombia,6.409276,-74.756698,COLOMBIA_ANTIOQUIA_CARACOLI,empty
1,caucasia,antioquia,colombia,7.987758,-75.198374,COLOMBIA_ANTIOQUIA_CAUCASIA,empty
2,el bagre,antioquia,colombia,7.697710,-74.622203,COLOMBIA_ANTIOQUIA_EL BAGRE,empty
3,mutata,antioquia,colombia,7.244068,-76.436542,COLOMBIA_ANTIOQUIA_MUTATA,empty
4,nechi,antioquia,colombia,8.093808,-74.775199,COLOMBIA_ANTIOQUIA_NECHI,empty
...,...,...,...,...,...,...,...
513,sopetran,antioquia,colombia,6.500958,-75.742498,COLOMBIA_ANTIOQUIA_SOPETRAN,empty
514,galan,santander,colombia,6.637910,-73.287453,COLOMBIA_SANTANDER_GALAN,empty
515,san miguel,santander,colombia,6.575929,-72.645549,COLOMBIA_SANTANDER_SAN MIGUEL,empty
516,teruel,huila,colombia,2.741633,-75.568345,COLOMBIA_HUILA_TERUEL,empty


In [13]:
# get euclidean distance between two points
def euclidean_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the Euclidean distance between two points on the earth (specified in decimal degrees)
    """
    # convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371  # Radius of earth in kilometers. Use 3956 for miles
    return c * r

# for each empty city get city with min euclidean distance to non-empty cities
def get_nearest_city(lat, lon, df_not_empty):
    """
    Get the nearest city from a list of cities
    """
    distances = []
    for index, row in df_not_empty.iterrows():
        distance = euclidean_distance(lat, lon, row['Latitude'], row['Longitude'])
        distances.append(distance)
    df_not_empty['distance'] = distances
    nearest_city = df_not_empty.loc[df_not_empty['distance'].idxmin()]
    return nearest_city

city = df_empty_col.iloc[0]
lat = city['Latitude']
lon = city['Longitude']

print(city)
get_nearest_city(lat, lon, df_not_empty)

city                            caracoli
state                          antioquia
country                         colombia
Latitude                        6.409276
Longitude                     -74.756698
idx_city     COLOMBIA_ANTIOQUIA_CARACOLI
status                             empty
Name: 0, dtype: object


city                            guatape
state                         antioquia
country                        colombia
Latitude                       6.234218
Longitude                    -75.161745
idx_city     COLOMBIA_ANTIOQUIA_GUATAPE
status                        not_empty
distance                      48.814342
Name: 746, dtype: object

In [14]:
nearest_cities = []
for i in range(len(df_empty_col)):
    city = df_empty_col.iloc[i]
    lat = city['Latitude']
    lon = city['Longitude']
    nearest_city = get_nearest_city(lat, lon, df_not_empty)
    nearest_cities.append(nearest_city['idx_city'])
    
    

In [16]:
df_empty_col['nearest_city'] = nearest_cities
df_empty_col

,city,state,country,Latitude,Longitude,idx_city,status,nearest_city
0,caracoli,antioquia,colombia,6.409276,-74.756698,COLOMBIA_ANTIOQUIA_CARACOLI,empty,COLOMBIA_ANTIOQUIA_GUATAPE
1,caucasia,antioquia,colombia,7.987758,-75.198374,COLOMBIA_ANTIOQUIA_CAUCASIA,empty,COLOMBIA_CORDOBA_SAN CARLOS
2,el bagre,antioquia,colombia,7.697710,-74.622203,COLOMBIA_ANTIOQUIA_EL BAGRE,empty,COLOMBIA_ANTIOQUIA_YONDO (CASABE)
3,mutata,antioquia,colombia,7.244068,-76.436542,COLOMBIA_ANTIOQUIA_MUTATA,empty,COLOMBIA_ANTIOQUIA_CHIGORODO
4,nechi,antioquia,colombia,8.093808,-74.775199,COLOMBIA_ANTIOQUIA_NECHI,empty,COLOMBIA_SUCRE_EL ROBLE
...,...,...,...,...,...,...,...,...
513,sopetran,antioquia,colombia,6.500958,-75.742498,COLOMBIA_ANTIOQUIA_SOPETRAN,empty,COLOMBIA_ANTIOQUIA_SAN JERONIMO
514,galan,santander,colombia,6.637910,-73.287453,COLOMBIA_SANTANDER_GALAN,empty,COLOMBIA_SANTANDER_ZAPATOCA
515,san miguel,santander,colombia,6.575929,-72.645549,COLOMBIA_SANTANDER_SAN MIGUEL,empty,COLOMBIA_SANTANDER_LOS SANTOS
516,teruel,huila,colombia,2.741633,-75.568345,COLOMBIA_HUILA_TERUEL,empty,COLOMBIA_HUILA_SANTA MARIA


In [21]:
#df_not_empty.drop(columns=['distance'], inplace=True)
df_not_empty['nearest_city'] = df_not_empty['idx_city']
df_not_empty

,city,state,country,Latitude,Longitude,idx_city,status,nearest_city
0,exterior_aruba,exterior,aruba,12.501363,-69.961848,ARUBA_EXTERIOR_EXTERIOR_ARUBA,not_empty,ARUBA_EXTERIOR_EXTERIOR_ARUBA
2,leticia,amazonas,colombia,-4.212921,-69.942596,COLOMBIA_AMAZONAS_LETICIA,not_empty,COLOMBIA_AMAZONAS_LETICIA
3,apartado,antioquia,colombia,7.884901,-76.622746,COLOMBIA_ANTIOQUIA_APARTADO,not_empty,COLOMBIA_ANTIOQUIA_APARTADO
5,carepa,antioquia,colombia,7.798452,-76.746039,COLOMBIA_ANTIOQUIA_CAREPA,not_empty,COLOMBIA_ANTIOQUIA_CAREPA
7,chigorodo,antioquia,colombia,7.612938,-76.638426,COLOMBIA_ANTIOQUIA_CHIGORODO,not_empty,COLOMBIA_ANTIOQUIA_CHIGORODO
...,...,...,...,...,...,...,...,...
779,cajibio,cauca,colombia,2.623169,-76.569350,COLOMBIA_CAUCA_CAJIBIO,not_empty,COLOMBIA_CAUCA_CAJIBIO
787,rionegro,antioquia,colombia,6.153617,-75.374169,COLOMBIA_ANTIOQUIA_RIONEGRO,not_empty,COLOMBIA_ANTIOQUIA_RIONEGRO
794,silvia,cauca,colombia,2.611544,-76.378424,COLOMBIA_CAUCA_SILVIA,not_empty,COLOMBIA_CAUCA_SILVIA
798,buenavista,quindio,colombia,4.359948,-75.738723,COLOMBIA_QUINDIO_BUENAVISTA,not_empty,COLOMBIA_QUINDIO_BUENAVISTA


In [22]:
df_full = pd.concat([df_empty_col, df_not_empty], axis=0)
df_full

,city,state,country,Latitude,Longitude,idx_city,status,nearest_city
0,caracoli,antioquia,colombia,6.409276,-74.756698,COLOMBIA_ANTIOQUIA_CARACOLI,empty,COLOMBIA_ANTIOQUIA_GUATAPE
1,caucasia,antioquia,colombia,7.987758,-75.198374,COLOMBIA_ANTIOQUIA_CAUCASIA,empty,COLOMBIA_CORDOBA_SAN CARLOS
2,el bagre,antioquia,colombia,7.697710,-74.622203,COLOMBIA_ANTIOQUIA_EL BAGRE,empty,COLOMBIA_ANTIOQUIA_YONDO (CASABE)
3,mutata,antioquia,colombia,7.244068,-76.436542,COLOMBIA_ANTIOQUIA_MUTATA,empty,COLOMBIA_ANTIOQUIA_CHIGORODO
4,nechi,antioquia,colombia,8.093808,-74.775199,COLOMBIA_ANTIOQUIA_NECHI,empty,COLOMBIA_SUCRE_EL ROBLE
...,...,...,...,...,...,...,...,...
779,cajibio,cauca,colombia,2.623169,-76.569350,COLOMBIA_CAUCA_CAJIBIO,not_empty,COLOMBIA_CAUCA_CAJIBIO
787,rionegro,antioquia,colombia,6.153617,-75.374169,COLOMBIA_ANTIOQUIA_RIONEGRO,not_empty,COLOMBIA_ANTIOQUIA_RIONEGRO
794,silvia,cauca,colombia,2.611544,-76.378424,COLOMBIA_CAUCA_SILVIA,not_empty,COLOMBIA_CAUCA_SILVIA
798,buenavista,quindio,colombia,4.359948,-75.738723,COLOMBIA_QUINDIO_BUENAVISTA,not_empty,COLOMBIA_QUINDIO_BUENAVISTA


In [23]:
df_full.to_csv('meteo_colombia/cities_dengue_with_nearest_ciy.csv', index=False)

In [25]:
df_not_empty.to_csv('meteo_colombia/cities_dengue_not_empty.csv', index=False)